In [3]:
import subprocess
import sys
import os

print("Menginstall library...")
!pip install -q python-terrier pyarabic pandas tashaphyne


if os.path.exists('skripsi-clir-code'):
    print("Repository ditemukan. Melakukan pull...")
    !cd skripsi-clir-code && git pull
else:
    print("Cloning repository...")
    !git clone https://github.com/syifaurrr/skripsi-clir-code.git

REPO_PATH = './skripsi-clir-code'
SRC_PATH  = os.path.join(REPO_PATH, 'src')
sys.path.append(SRC_PATH)
    
# ==========================================
# CELL: RRF FUSION BM25+RM3 + DENSE RETRIEVER
# ==========================================
import pandas as pd
import numpy as np
import pyterrier as pt
from collections import defaultdict
from IPython.display import display
from arabic_preprocessing import preprocess_pipeline
# 3. Install FAISS (Coba GPU dulu, kalau gagal pakai CPU)
try:
    if gpu_available:
        print("Mencoba install faiss-gpu...")
        # Kadang faiss-gpu rewel versi, kita coba install
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-gpu"])
        print("✅ faiss-gpu berhasil diinstall.")
    else:
        raise Exception("GPU off")
except:
    print("⚠️ Gagal install faiss-gpu (atau GPU mati). Menginstall faiss-cpu...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
    print("✅ faiss-cpu berhasil diinstall (Retrieval tetap cepat, Encoding yang butuh GPU).")




# --- 1. Konfigurasi Parameter ---
# BM25+RM3 parameters (best dari skenario 1)
BM25_K1 = 0.5
BM25_B = 1.0
RM3_FB_DOCS = 10
RM3_FB_TERMS = 20
RM3_FB_LAMBDA = 0.8

# Dense Retriever parameters
DENSE_MODEL_PATH = '/kaggle/input/datasets/syifaur/mmbert-base-bs8-ep6-lr3e-05-wr0-1-seq128'

# RRF Parameter
RRF_K = 60  # Konstanta RRF (biasanya 60, bisa dicoba 30-100)

# --- 2. Load Data ---
DATA_PATH = './skripsi-clir-code/data'
raw_docs_path = os.path.join(DATA_PATH, 'raw/fathul_muin.csv')
queries_indo_path = os.path.join(DATA_PATH, 'queries/queries_indo.csv')
queries_arab_path = os.path.join(DATA_PATH, 'queries/queries_arab.csv')
qrels_path = os.path.join(DATA_PATH, 'queries/qrels.csv')

print("Loading data...")
df_docs = pd.read_csv(raw_docs_path)
df_docs['docno'] = df_docs['docno'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

# Preprocess dokumen untuk BM25 (Arab)
from arabic_preprocessing import preprocess_pipeline
df_docs['text_processed'] = df_docs['text'].apply(preprocess_pipeline)

# Preprocess dokumen untuk Dense (normalisasi sederhana)
def normalize_arabic(text):
    if not isinstance(text, str): return ""
    text = araby.strip_tashkeel(text)
    text = araby.strip_tatweel(text)
    text = araby.normalize_alef(text)
    text = re.sub(r'[ى]', 'ي', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)
    return text

df_docs['text_clean'] = df_docs['text'].astype(str).apply(normalize_arabic)

# Load queries: INDO untuk Dense, ARAB untuk BM25
df_queries_indo = pd.read_csv(queries_indo_path)
df_queries_indo['qid'] = df_queries_indo['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

df_queries_arab = pd.read_csv(queries_arab_path)
df_queries_arab['qid'] = df_queries_arab['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

# Load qrels
qrels = pd.read_csv(qrels_path)
qrels['qid'] = qrels['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
qrels['docno'] = qrels['docno'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
qrels['label'] = qrels['label'].astype(int)

print(f"✅ Data loaded:")
print(f"   Docs: {len(df_docs)}")
print(f"   Queries (Indo): {len(df_queries_indo)}")
print(f"   Queries (Arab): {len(df_queries_arab)}")
print(f"   Qrels: {len(qrels)}")

# --- 3. Preprocess Queries untuk BM25 (Arab) ---
print("\n" + "="*60)
print("Preprocessing Arabic Queries for BM25+RM3...")
print("="*60)

# Gunakan query_llm (Gemini) - performa terbaik dari skenario 1
df_queries_arab_bm25 = df_queries_arab[['qid', 'query_llm']].rename(columns={'query_llm': 'query'}).copy()

# Preprocess query Arab
df_queries_arab_bm25['query'] = df_queries_arab_bm25['query'].astype(str).apply(preprocess_pipeline)
df_queries_arab_bm25['query'] = df_queries_arab_bm25['query'].str.replace(r'[^\w\s]', '', regex=True)

# Filter query kosong
before = len(df_queries_arab_bm25)
df_queries_arab_bm25 = df_queries_arab_bm25[df_queries_arab_bm25['query'].notna() & (df_queries_arab_bm25['query'].str.strip() != "")]
print(f"   Valid queries: {len(df_queries_arab_bm25)} / {before}")

# --- 4. Preprocess Queries untuk Dense (Indo) ---
print("\n" + "="*60)
print("Preprocessing Indonesian Queries for Dense Retriever...")
print("="*60)

df_queries_dense = df_queries_indo[['qid', 'query']].copy()
print(f"   Queries: {len(df_queries_dense)}")

# Filter qrels hanya untuk qid yang valid
valid_qids = set(df_queries_arab_bm25['qid']) & set(df_queries_dense['qid'])
qrels_filtered = qrels[qrels['qid'].isin(valid_qids)].copy()
print(f"   Valid Qrels: {len(qrels_filtered)}")

# --- 5. Build BM25 Index (Sekali saja, reuse dari skenario 1) ---
print("\n" + "="*60)
print("Building/Loading BM25 Index...")
print("="*60)

index_dir = os.path.join(DATA_PATH, 'indices/skenario_1_sparse')

if os.path.exists(index_dir):
    print(f"Loading existing index from {index_dir}...")
    index_ref = pt.IndexRef.of(index_dir)
    index = pt.IndexFactory.of(index_ref)
    print(f"✅ Index loaded: {index.getCollectionStatistics().getNumberOfDocuments()} docs")
else:
    print("Creating new BM25 index...")
    os.makedirs(index_dir, exist_ok=True)
    indexer = pt.IterDictIndexer(
        index_dir,
        meta={'docno': 20, 'text': 4096},
        overwrite=True,
        stemmer=None,
        stopwords=None,
        tokeniser="UTFTokeniser"
    )
    docs_to_index = df_docs[['docno', 'text_processed']].rename(columns={'text_processed': 'text'})
    index_ref = indexer.index(docs_to_index.to_dict(orient='records'))
    index = pt.IndexFactory.of(index_ref)
    print(f"✅ Index created: {index.getCollectionStatistics().getNumberOfDocuments()} docs")

# --- 6. Build BM25+RM3 Retriever (menggunakan query Arab) ---
print("\n" + "="*60)
print("Building BM25+RM3 Retriever (Arabic Queries)...")
print("="*60)

bm25 = pt.terrier.Retriever(
    index_ref,
    wmodel="BM25",
    controls={"bm25.k_1": BM25_K1, "bm25.b": BM25_B},
    properties={"termpipelines": ""}
)

# BM25+RM3 pipeline
rm3_pipeline = bm25 >> pt.rewrite.RM3(
    index_ref,
    fb_docs=RM3_FB_DOCS,
    fb_terms=RM3_FB_TERMS,
    fb_lambda=RM3_FB_LAMBDA
) >> bm25

print(f"✅ BM25+RM3 ready (k1={BM25_K1}, b={BM25_B}, fb_docs={RM3_FB_DOCS}, fb_terms={RM3_FB_TERMS}, λ={RM3_FB_LAMBDA})")

# --- 7. Build Dense Retriever (menggunakan query Indo) ---
print("\n" + "="*60)
print("Building Dense Retriever (Indonesian Queries)...")
print("="*60)

from sentence_transformers import SentenceTransformer
import faiss

class FaissRetriever(pt.Transformer):
    def __init__(self, model_name, doc_df=None, batch_size=32):
        self.model = SentenceTransformer(model_name)
        
        # Patch untuk tokenizer
        if hasattr(self.model, 'tokenizer') and 'token_type_ids' in self.model.tokenizer.model_input_names:
            self.model.tokenizer.model_input_names.remove('token_type_ids')
            
        self.index = None
        self.doc_map = {}
        self.batch_size = batch_size
        
        if doc_df is not None:
            self._index_docs(doc_df)
    
    def _index_docs(self, doc_df):
        print(f"Encoding {len(doc_df)} documents...")
        docs = doc_df['text_clean'].tolist()
        docnos = doc_df['docno'].tolist()
        
        embeddings = self.model.encode(docs, batch_size=self.batch_size, show_progress_bar=True, convert_to_numpy=True)
        faiss.normalize_L2(embeddings)
        
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)
        
        self.doc_map = {i: docno for i, docno in enumerate(docnos)}
        print(f"✅ Indexing complete: {len(self.doc_map)} docs")
    
    def transform(self, queries):
        q_texts = queries['query'].tolist()
        q_embeddings = self.model.encode(q_texts, batch_size=self.batch_size, show_progress_bar=False, convert_to_numpy=True)
        faiss.normalize_L2(q_embeddings)
        
        k = 1000
        scores, ids = self.index.search(q_embeddings, k)
        
        results = []
        for i, qid in enumerate(queries['qid']):
            for j in range(k):
                idx = ids[i][j]
                if idx != -1:
                    results.append({
                        'qid': str(qid),
                        'docno': self.doc_map[idx],
                        'score': float(scores[i][j]),
                        'rank': j + 1
                    })
        return pd.DataFrame(results)

dense_retriever = FaissRetriever(
    model_name=DENSE_MODEL_PATH,
    doc_df=df_docs[['docno', 'text_clean']],
    batch_size=32
)
print(f"✅ Dense Retriever ready")

# --- 8. Get Retrieval Results ---
print("\n" + "="*60)
print("Running Retrieval for All Queries...")
print("="*60)

# BM25+RM3 dengan query Arab
print("Running BM25+RM3 retrieval (Arabic queries)...")
rm3_results = rm3_pipeline.transform(df_queries_arab_bm25)
rm3_results = rm3_results[['qid', 'docno', 'rank']].copy()
rm3_results = rm3_results[rm3_results['qid'].isin(valid_qids)]
print(f"   ✅ BM25+RM3: {len(rm3_results)} results for {len(rm3_results['qid'].unique())} queries")

# Dense Retriever dengan query Indo
print("Running Dense retrieval (Indonesian queries)...")
dense_results = dense_retriever.transform(df_queries_dense)
dense_results = dense_results[['qid', 'docno', 'rank']].copy()
dense_results = dense_results[dense_results['qid'].isin(valid_qids)]
print(f"   ✅ mmBERT-base Fine-Tuned (JH-POLO): {len(dense_results)} results for {len(dense_results['qid'].unique())} queries")

# --- 9. RRF Fusion Function ---
def reciprocal_rank_fusion(results_list, k=60):
    fusion_scores = defaultdict(lambda: defaultdict(float))
    
    for model_idx, results in enumerate(results_list):
        for _, row in results.iterrows():
            qid = row['qid']
            docno = row['docno']
            rank = row['rank']
            rrf_score = 1.0 / (k + rank)
            fusion_scores[qid][docno] += rrf_score
    
    fused_results = []
    for qid, doc_scores in fusion_scores.items():
        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        for rank, (docno, score) in enumerate(sorted_docs, 1):
            fused_results.append({
                'qid': qid,
                'docno': docno,
                'score': score,
                'rank': rank
            })
    
    return pd.DataFrame(fused_results)

# --- 10. Apply RRF ---
print("\n" + "="*60)
print(f"Applying RRF Fusion (k={RRF_K})...")
print("="*60)

fused_results = reciprocal_rank_fusion([rm3_results, dense_results], k=RRF_K)
print(f"✅ Fusion complete: {len(fused_results)} results for {len(fused_results['qid'].unique())} queries")

# --- 11. Evaluate Individual Models ---
print("\n" + "="*60)
print("Evaluating Individual Models...")
print("="*60)

eval_metrics = ["recip_rank", "success_10", "success_20", "success_50", "success_100"]

# Evaluasi BM25+RM3
print("Evaluating BM25+RM3...")
rm3_run = rm3_results.copy()
rm3_run['score'] = 1.0 / rm3_run['rank']  # Assign dummy score

# Filter qrels untuk qid yang ada di queries
qrels_for_eval = qrels_filtered

rm3_eval = pt.Experiment(
    [rm3_run],
    df_queries_dense[df_queries_dense['qid'].isin(valid_qids)],
    qrels_for_eval,
    eval_metrics=eval_metrics,
    names=["BM25+RM3"],
    validate="ignore"
)

# Evaluasi Dense
print("Evaluating mmBERT-base Fine-Tuned (JH-POLO)...")
dense_run = dense_results.copy()
dense_run['score'] = 1.0 / dense_run['rank']

dense_eval = pt.Experiment(
    [dense_run],
    df_queries_dense[df_queries_dense['qid'].isin(valid_qids)],
    qrels_for_eval,
    eval_metrics=eval_metrics,
    names=["mmBERT-base Fine-Tuned (JH-POLO)"],
    validate="ignore"
)

# Evaluasi RRF
print("Evaluating RRF Fusion...")
rrf_run = fused_results.copy()
rrf_run['score'] = rrf_run['score'].astype(float)

rrf_eval = pt.Experiment(
    [rrf_run],
    df_queries_dense[df_queries_dense['qid'].isin(valid_qids)],
    qrels_for_eval,
    eval_metrics=eval_metrics,
    names=["RRF (BM25+RM3 + mmBERT)"],
    validate="ignore"
)

# --- 12. Comparison Table ---
print("\n" + "="*60)
print("📊 COMPARISON: Individual Models vs RRF Fusion")
print("="*60)

comparison = pd.concat([rm3_eval, dense_eval, rrf_eval], ignore_index=True)

# Format persentase
for k in [10, 20, 50, 100]:
    comparison[f'success_{k} (%)'] = (comparison[f'success_{k}'] * 100).round(2)
comparison['recip_rank'] = comparison['recip_rank'].round(4)

display(comparison[['name', 'recip_rank', 'success_10 (%)', 'success_20 (%)', 'success_50 (%)', 'success_100 (%)']])

# --- 13. Cek Coverage Queries ---
print("\n" + "="*60)
print("Query Coverage Check:")
print("="*60)
print(f"BM25+RM3 queries: {len(rm3_results['qid'].unique())} queries")
print(f"mmBERT-base Fine-Tuned (JH-POLO) queries: {len(dense_results['qid'].unique())} queries")
print(f"RRF queries: {len(fused_results['qid'].unique())} queries")
print(f"Qrels queries: {len(qrels_for_eval['qid'].unique())} queries")
print(f"Total unique queries: {len(valid_qids)}")

# --- 14. Save Results ---
output_dir = os.path.join(DATA_PATH, 'results', 'rrf_fusion')
os.makedirs(output_dir, exist_ok=True)

comparison.to_csv(os.path.join(output_dir, 'rrf_comparison.csv'), index=False)
fused_results.to_csv(os.path.join(output_dir, 'rrf_fused_run.csv'), index=False)

# Simpan juga runs individual
rm3_results.to_csv(os.path.join(output_dir, 'bm25_rm3_run.csv'), index=False)
dense_results.to_csv(os.path.join(output_dir, 'mmbert_finetuned_run.csv'), index=False)

print("\n" + "="*60)
print("✅ RRF FUSION COMPLETE!")
print(f"📁 Results saved to: {output_dir}")
print("   - rrf_comparison.csv")
print("   - rrf_fused_run.csv")
print("   - bm25_rm3_run.csv")
print("   - mmbert_finetuned_run.csv")
print("="*60)

# --- 15. Bonus: Hit Rate Analysis ---
print("\n" + "="*60)
print("📈 HIT RATE ANALYSIS (Success@k)")
print("="*60)

def calculate_hit_rate(results_df, qrels_df, k_values=[10, 20, 50, 100]):
    """Hitung hit rate per query untuk berbagai k"""
    # Ambil hanya dokumen relevan (label >= 1)
    relevant_docs = qrels_df[qrels_df['label'] >= 1][['qid', 'docno']].drop_duplicates()
    
    hit_rates = {}
    for k in k_values:
        hits = 0
        total = 0
        
        for qid in results_df['qid'].unique():
            q_results = results_df[results_df['qid'] == qid].head(k)
            q_relevant = relevant_docs[relevant_docs['qid'] == qid]['docno'].tolist()
            
            if len(q_relevant) > 0:
                total += 1
                if set(q_results['docno']).intersection(set(q_relevant)):
                    hits += 1
        
        hit_rates[k] = (hits / total * 100) if total > 0 else 0
    
    return hit_rates

print("\nBM25+RM3 Hit Rates:")
rm3_hits = calculate_hit_rate(rm3_results, qrels_for_eval)
for k, rate in rm3_hits.items():
    print(f"   Success@{k}: {rate:.2f}%")

print("\nmmBERT-base Fine-Tuned (JH-POLO) Hit Rates:")
dense_hits = calculate_hit_rate(dense_results, qrels_for_eval)
for k, rate in dense_hits.items():
    print(f"   Success@{k}: {rate:.2f}%")

print("\nRRF Fusion Hit Rates:")
rrf_hits = calculate_hit_rate(fused_results, qrels_for_eval)
for k, rate in rrf_hits.items():
    print(f"   Success@{k}: {rate:.2f}%")

Menginstall library...
Repository ditemukan. Melakukan pull...
Already up to date.
⚠️ Gagal install faiss-gpu (atau GPU mati). Menginstall faiss-cpu...
✅ faiss-cpu berhasil diinstall (Retrieval tetap cepat, Encoding yang butuh GPU).
Loading data...
✅ Data loaded:
   Docs: 639
   Queries (Indo): 153
   Queries (Arab): 153
   Qrels: 153

Preprocessing Arabic Queries for BM25+RM3...
   Valid queries: 153 / 153

Preprocessing Indonesian Queries for Dense Retriever...
   Queries: 153
   Valid Qrels: 153

Building/Loading BM25 Index...
Loading existing index from ./skripsi-clir-code/data/indices/skenario_1_sparse...
✅ Index loaded: 639 docs

Building BM25+RM3 Retriever (Arabic Queries)...
✅ BM25+RM3 ready (k1=0.5, b=1.0, fb_docs=10, fb_terms=20, λ=0.8)

Building Dense Retriever (Indonesian Queries)...


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Encoding 639 documents...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

✅ Indexing complete: 639 docs
✅ Dense Retriever ready

Running Retrieval for All Queries...
Running BM25+RM3 retrieval (Arabic queries)...
   ✅ BM25+RM3: 88119 results for 153 queries
Running Dense retrieval (Indonesian queries)...
   ✅ mmBERT-base Fine-Tuned (JH-POLO): 97767 results for 153 queries

Applying RRF Fusion (k=60)...
✅ Fusion complete: 97767 results for 153 queries

Evaluating Individual Models...
Evaluating BM25+RM3...
Evaluating mmBERT-base Fine-Tuned (JH-POLO)...
Evaluating RRF Fusion...

📊 COMPARISON: Individual Models vs RRF Fusion


,name,recip_rank,success_10 (%),success_20 (%),success_50 (%),success_100 (%)
0,BM25+RM3,0.3602,56.86,66.67,82.35,90.20
1,mmBERT-base Fine-Tuned (JH-POLO),0.2976,50.98,59.48,67.32,84.31
2,RRF (BM25+RM3 + mmBERT),0.3936,64.71,83.66,94.12,97.39



Query Coverage Check:
BM25+RM3 queries: 153 queries
mmBERT-base Fine-Tuned (JH-POLO) queries: 153 queries
RRF queries: 153 queries
Qrels queries: 153 queries
Total unique queries: 153

✅ RRF FUSION COMPLETE!
📁 Results saved to: ./skripsi-clir-code/data/results/rrf_fusion
   - rrf_comparison.csv
   - rrf_fused_run.csv
   - bm25_rm3_run.csv
   - mmbert_finetuned_run.csv

📈 HIT RATE ANALYSIS (Success@k)

BM25+RM3 Hit Rates:
   Success@10: 56.86%
   Success@20: 66.67%
   Success@50: 82.35%
   Success@100: 90.20%

mmBERT-base Fine-Tuned (JH-POLO) Hit Rates:
   Success@10: 50.98%
   Success@20: 59.48%
   Success@50: 67.32%
   Success@100: 84.31%

RRF Fusion Hit Rates:
   Success@10: 64.71%
   Success@20: 83.66%
   Success@50: 94.12%
   Success@100: 97.39%
